# Hierarchical fusion with sentiment-aware CLIP attributes



In [1]:
%pip install transformers==4.57.1 tqdm==4.67.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 44.1 MB/s  0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers]
ERROR: pip's dependen

In [2]:
import ast, csv, hashlib, json, os, pickle, platform, random, time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from PIL import Image, __version__ as pillow_version
import transformers
from transformers import CLIPModel, CLIPProcessor

DATA_ROOT = Path('/kaggle/input/datasets/fruich/mmsd-full/data')
ASSET_ROOT = DATA_ROOT / 'raw/mmsd'
FEATURE_DIR = Path('/kaggle/input/datasets/fruich/mmsd-full/data/features/resnetv2_50_a1h_in1k_finetuned')
IMAGE_DIR = DATA_ROOT / 'raw/dataset_image'
CLIP_ATTRIBUTE_DIR = Path('/kaggle/working/features/clip_sentiment_attributes')
ATTRIBUTE_PREDICTIONS = CLIP_ATTRIBUTE_DIR / 'extract_all'
CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'
CLIP_REVISION = '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268'
CLIP_PROMPT = 'a photo of {word}.'
SENTIMENT_WORDS = ('good', 'bad', 'happy', 'sad', 'angry', 'calm', 'beautiful', 'ugly')
SENTIMENT_PROMPT = 'a photo of a scene that feels {word}.'
NEUTRAL_PROMPT = 'a photo of a scene that feels neutral.'
CLIP_BATCH_SIZE = 64
SEEDS, BATCH_SIZE, EPOCHS, PATIENCE = (42, 43, 44), 32, 8, 5
LEARNING_RATE, L2, SEQ_LEN, NUM_WORKERS = 0.001, 1e-7, 75, 4
RUN_ROOT = Path('/kaggle/working/runs/hierarchical_finetuned_clip_sentiment_packed')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
print('PyTorch:', torch.__version__, 'Device:', DEVICE)


I0000 00:00:1794676441.586686      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


PyTorch: 2.11.0+cu130 Device: cuda


## Extend sentiment vocabulary using existing text GloVe vectors

In [3]:
class VocabularyLoader(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError('Global references are not allowed in vocabularies')

def load_vocabulary(path):
    with path.open('rb') as stream:
        return VocabularyLoader(stream, encoding='latin1').load()

word_vocab = load_vocabulary(ASSET_ROOT / 'words/vocab')
attribute_vocab = load_vocabulary(ASSET_ROOT / 'ExtractWords/vocab')
word_weights = torch.from_numpy(np.loadtxt(ASSET_ROOT / 'words/vector', dtype=np.float32))
attribute_weights = torch.from_numpy(np.loadtxt(ASSET_ROOT / 'ExtractWords/vector', dtype=np.float32))
assert word_vocab['<pad>'] == 0
assert attribute_weights.shape == (len(attribute_vocab), 200)
assert word_weights.shape == (len(word_vocab), 200)
blocked = {'sarcasm', 'sarcastic', 'reposting', '<url>', 'joke', 'humour',
           'humor', 'jokes', 'irony', 'ironic', 'exgag'}
STOPWORDS = set('a an the of with in on and is it are was were be been being to from by for at as into over under after before between through during without within about against among around across up down out off then than that this these those i me my we our you your he she they them his her their its who whom which what where when why how all any both each few more most other some such no nor not only own same so too very can will just should could would do does did have has had here there again further once'.split())

def extend_attribute_embeddings(vocabulary, weights, text_vocabulary, text_weights, sentiment_words):
    if weights.shape != (len(vocabulary), 200) or text_weights.shape != (len(text_vocabulary), 200):
        raise ValueError('Vocabulary and 200-dimensional embedding shapes must match')
    if set(vocabulary.values()) != set(range(len(vocabulary))):
        raise ValueError('Attribute vocabulary indices must be contiguous')
    if len(set(sentiment_words)) != len(sentiment_words):
        raise ValueError('Sentiment words must be unique')
    additions = [word for word in sentiment_words if word not in vocabulary]
    unsupported = [word for word in additions if word not in text_vocabulary]
    if unsupported: raise ValueError(f'No existing GloVe vectors for: {unsupported}')
    expanded = dict(vocabulary)
    vectors = []
    for word in additions:
        vector = text_weights[text_vocabulary[word]].detach().clone()
        if vector.shape != (200,) or not torch.isfinite(vector).all() or vector.norm() == 0:
            raise ValueError(f'Invalid sentiment embedding: {word}')
        expanded[word] = len(expanded)
        vectors.append(vector)
    expanded_weights = torch.cat([weights, torch.stack(vectors)]) if vectors else weights
    assert torch.equal(expanded_weights[:len(vocabulary)], weights)
    return expanded, expanded_weights, additions

attribute_vocab, attribute_weights, added_sentiment_words = extend_attribute_embeddings(
    attribute_vocab, attribute_weights, word_vocab, word_weights, SENTIMENT_WORDS)
assert all(word in attribute_vocab for word in SENTIMENT_WORDS)

def select_candidates(vocabulary):
    candidates = sorted(word for word in vocabulary
                        if word.isascii() and word.isalpha() and word.islower()
                        and len(word) > 1 and word not in STOPWORDS and word not in SENTIMENT_WORDS)
    if len(candidates) < 5:
        raise ValueError('Need at least five supported CLIP candidate words')
    return candidates

def required_tweet_ids(asset_root, splits=('train', 'valid2', 'test2')):
    tweet_ids = set()
    for split in splits:
        rows = (ast.literal_eval(line) for line in
                (asset_root / f'text/{split}.txt').read_text().splitlines() if line.strip())
        tweet_ids.update(str(row[0]) for row in rows if not blocked.intersection(row[1].split()))
    return sorted(tweet_ids)

candidates = select_candidates(attribute_vocab)
def select_attributes(descriptive_top5, sentiment_scores):
    if len(descriptive_top5) != 5 or len({word for word, _ in descriptive_top5}) != 5:
        raise ValueError('Expected five distinct descriptive candidates')
    if len(sentiment_scores) != 1 + len(SENTIMENT_WORDS) or not all(np.isfinite(sentiment_scores)):
        raise ValueError('Expected finite neutral plus sentiment similarities')
    winner = max(range(len(sentiment_scores)), key=lambda index: sentiment_scores[index])
    if winner == 0:
        return descriptive_top5, 'neutral'
    word = SENTIMENT_WORDS[winner - 1]
    if any(candidate == word for candidate, _ in descriptive_top5):
        raise ValueError('Sentiment word must not appear in descriptive candidates')
    return [*descriptive_top5[:4], (word, sentiment_scores[winner])], word

print('Candidate count:', len(candidates))
print('Sentiment words:', SENTIMENT_WORDS)
print('New attribute rows initialized from text GloVe:', added_sentiment_words)
print('Attribute embedding shape:', tuple(attribute_weights.shape))


Candidate count: 996
Sentiment words: ('good', 'bad', 'happy', 'sad', 'angry', 'calm', 'beautiful', 'ugly')
New attribute rows initialized from text GloVe: ['good', 'bad', 'sad', 'angry', 'calm', 'ugly']
Attribute embedding shape: (1008, 200)


## Frozen CLIP extraction (once; no fitting or test-label use)
Refuses to overwrite extract_all. If extraction already completed in this session, skip this cell when rerunning fusion-only cells. The following cell checks the saved manifest and file hash.


In [4]:
if ATTRIBUTE_PREDICTIONS.exists():
    raise FileExistsError(f'Existing CLIP predictions: {ATTRIBUTE_PREDICTIONS}; skip extraction to reuse them')
tweet_ids = required_tweet_ids(ASSET_ROOT)
if not tweet_ids: raise ValueError('No retained images found in MMSD splits')
image_paths = [IMAGE_DIR / f'{tweet_id}.jpg' for tweet_id in tweet_ids]
missing_images = [str(path) for path in image_paths if not path.is_file()]
if missing_images: raise FileNotFoundError(f'Missing {len(missing_images)} images; first: {missing_images[0]}')
CLIP_ATTRIBUTE_DIR.mkdir(parents=True, exist_ok=True)
start = time.perf_counter()
if DEVICE.type == 'cuda': torch.cuda.reset_peak_memory_stats(DEVICE)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID, revision=CLIP_REVISION).eval().to(DEVICE)
clip_model.requires_grad_(False)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, revision=CLIP_REVISION, use_fast=False)
rows, score_rows, sentiment_rows = [], [], []
all_prompts = [CLIP_PROMPT.format(word=word) for word in candidates]
all_prompts += [NEUTRAL_PROMPT, *[SENTIMENT_PROMPT.format(word=word) for word in SENTIMENT_WORDS]]
with torch.inference_mode():
    text_vectors = []
    for start_index in range(0, len(all_prompts), CLIP_BATCH_SIZE):
        prompts = all_prompts[start_index:start_index + CLIP_BATCH_SIZE]
        inputs = clip_processor(text=prompts, padding=True, truncation=True, return_tensors='pt')
        vectors = clip_model.get_text_features(**{key: value.to(DEVICE) for key, value in inputs.items()})
        vectors = vectors / vectors.norm(dim=-1, keepdim=True)
        if not torch.isfinite(vectors).all(): raise FloatingPointError('Non-finite CLIP text features')
        text_vectors.append(vectors)
    text_vectors = torch.cat(text_vectors)
    for start_index in tqdm(range(0, len(image_paths), CLIP_BATCH_SIZE), desc='Extract CLIP attributes'):
        paths = image_paths[start_index:start_index + CLIP_BATCH_SIZE]
        images = []
        for path in paths:
            with Image.open(path) as image: images.append(image.convert('RGB'))
        pixels = clip_processor(images=images, return_tensors='pt')['pixel_values'].to(DEVICE)
        image_vectors = clip_model.get_image_features(pixel_values=pixels)
        image_vectors = image_vectors / image_vectors.norm(dim=-1, keepdim=True)
        scores = (image_vectors @ text_vectors.T).cpu()
        if not torch.isfinite(scores).all(): raise FloatingPointError('Non-finite CLIP similarities')
        descriptive_scores = scores[:, :len(candidates)]
        sentiment_scores = scores[:, len(candidates):].tolist()
        top_ids = torch.argsort(descriptive_scores, dim=1, descending=True, stable=True)[:, :5].tolist()
        for i, (path, indices) in enumerate(zip(paths, top_ids)):
            descriptive_top5 = [(candidates[j], float(scores[i, j])) for j in indices]
            selected, sentiment = select_attributes(descriptive_top5, sentiment_scores[i])
            words = [word for word, _ in selected]
            assert len(words) == len(set(words)) == 5
            assert all(word in attribute_vocab for word in words)
            rows.append([path.stem, *words])
            score_rows.extend((path.stem, rank, word, score,
                               'sentiment' if sentiment != 'neutral' and rank == 5 else 'descriptive')
                              for rank, (word, score) in enumerate(selected, 1))
            sentiment_rows.append((path.stem, sentiment, *sentiment_scores[i]))
extraction_seconds = time.perf_counter() - start
assert len(rows) == len(tweet_ids) and {row[0] for row in rows} == set(tweet_ids)
manifest = {'model_id': CLIP_MODEL_ID, 'revision': CLIP_REVISION, 'frozen': True,
            'prompt': CLIP_PROMPT, 'sentiment_prompt': SENTIMENT_PROMPT, 'neutral_prompt': NEUTRAL_PROMPT,
            'candidate_policy': 'original attribute vocabulary minus fixed sentiment words, stopwords, and non-lowercase/non-alphabetic/single-character tokens',
            'selection_policy': 'four descriptive plus one sentiment; five descriptive when neutral wins or ties',
            'candidates': candidates, 'excluded_stopwords': sorted(STOPWORDS), 'sentiment_words': list(SENTIMENT_WORDS),
            'sentiment_score_order': ['neutral', *SENTIMENT_WORDS],
            'expanded_attribute_vocab': attribute_vocab, 'added_sentiment_words': added_sentiment_words,
            'added_embedding_source': 'existing words/vector rows, indexed by words/vocab; no random initialization',
            'embedding_shape': list(attribute_weights.shape),
            'initial_embedding_sha256': hashlib.sha256(attribute_weights.numpy().tobytes()).hexdigest(),
            'text_vectors_sha256': hashlib.sha256((ASSET_ROOT / 'words/vector').read_bytes()).hexdigest(),
            'image_dir': str(IMAGE_DIR), 'images': len(rows), 'batch_size': CLIP_BATCH_SIZE,
            'ranking': 'separate descriptive and sentiment cosine rankings; stable descriptive ties; neutral-first sentiment ties; no cross-pool score threshold',
            'image_preprocessing': clip_processor.image_processor.to_dict(),
            'transformers': transformers.__version__, 'torch': torch.__version__, 'pillow': pillow_version,
            'device': str(DEVICE), 'extraction_seconds_including_model_load': extraction_seconds,
            'peak_allocated_gpu_bytes': torch.cuda.max_memory_allocated(DEVICE) if DEVICE.type == 'cuda' else None,
            'attribute_vocab_sha256': hashlib.sha256((ASSET_ROOT / 'ExtractWords/vocab').read_bytes()).hexdigest()}
temporary = ATTRIBUTE_PREDICTIONS.with_suffix('.tmp')
temporary.write_text(''.join(repr(row) + '\n' for row in rows))
manifest['predictions_sha256'] = hashlib.sha256(temporary.read_bytes()).hexdigest()
with (CLIP_ATTRIBUTE_DIR / 'top5_scores.csv').open('w', newline='') as stream:
    writer = csv.writer(stream)
    writer.writerow(['tweet_id', 'rank', 'attribute', 'cosine_similarity', 'attribute_source'])
    writer.writerows(score_rows)
with (CLIP_ATTRIBUTE_DIR / 'sentiment_scores.csv').open('w', newline='') as stream:
    writer = csv.writer(stream)
    writer.writerow(['tweet_id', 'selected_sentiment_or_neutral', 'neutral', *SENTIMENT_WORDS])
    writer.writerows(sentiment_rows)
(CLIP_ATTRIBUTE_DIR / 'attribute_vocab.json').write_text(json.dumps(attribute_vocab, indent=2) + '\n')
np.save(CLIP_ATTRIBUTE_DIR / 'attribute_embedding_init.npy', attribute_weights.numpy(), allow_pickle=False)
(CLIP_ATTRIBUTE_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
os.replace(temporary, ATTRIBUTE_PREDICTIONS)
print('Extraction seconds (including model load):', extraction_seconds)
preview_ids = set(required_tweet_ids(ASSET_ROOT, ('train', 'valid2')))
print('Training/validation examples:', [row for row in rows if row[0] in preview_ids][:10])
print('Training/validation neutral fallbacks:', sum(row[0] in preview_ids and row[1] == 'neutral'
                                                  for row in sentiment_rows), 'of', len(preview_ids))
del clip_model, clip_processor, text_vectors, vectors, image_vectors, scores, descriptive_scores, pixels, inputs, rows, score_rows, sentiment_rows
if DEVICE.type == 'cuda': torch.cuda.empty_cache()


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Extract CLIP attributes:   0%|          | 0/385 [00:00<?, ?it/s]

Extraction seconds (including model load): 2261.728407259
Training/validation examples: [['682721949072625664', 'gold', 'getting', 'cooked', 'giving', 'bad'], ['682722242111774720', 'passengers', 'driving', 'sailing', 'passing', 'bad'], ['682727470433308672', 'course', 'like', 'someones', 'trying', 'bad'], ['682751920956641280', 'eyes', 'purple', 'hands', 'look', 'angry'], ['682762265041321989', 'country', 'road', 'public', 'says', 'ugly'], ['682773510196822016', 'space', 'race', 'day', 'someones', 'angry'], ['682783114343518208', 'birthday', 'party', 'cats', 'course', 'angry'], ['682784347305410560', 'drinking', 'umpire', 'tea', 'lying', 'like'], ['682836685323956224', 'using', 'number', 'includes', 'prepared', 'ugly'], ['682864661813981184', 'shopping', 'foods', 'eating', 'sandwiches', 'angry']]
Training/validation neutral fallbacks: 3006 of 22226


## Load CLIP words; identical fusion dataset and loaders


In [5]:
attribute_manifest = json.loads((CLIP_ATTRIBUTE_DIR / 'manifest.json').read_text())
assert attribute_manifest['model_id'] == CLIP_MODEL_ID and attribute_manifest['revision'] == CLIP_REVISION
assert attribute_manifest['prompt'] == CLIP_PROMPT and attribute_manifest['candidates'] == candidates
assert attribute_manifest['sentiment_prompt'] == SENTIMENT_PROMPT and attribute_manifest['neutral_prompt'] == NEUTRAL_PROMPT
assert attribute_manifest['sentiment_words'] == list(SENTIMENT_WORDS)
assert attribute_manifest['expanded_attribute_vocab'] == attribute_vocab
assert attribute_manifest['initial_embedding_sha256'] == hashlib.sha256(attribute_weights.numpy().tobytes()).hexdigest()
assert attribute_manifest['attribute_vocab_sha256'] == hashlib.sha256((ASSET_ROOT / 'ExtractWords/vocab').read_bytes()).hexdigest()
assert attribute_manifest['predictions_sha256'] == hashlib.sha256(ATTRIBUTE_PREDICTIONS.read_bytes()).hexdigest()
attributes = {str(row[0]): row[1:] for row in
              (ast.literal_eval(line) for line in ATTRIBUTE_PREDICTIONS.read_text().splitlines() if line.strip())}
assert set(attributes) == set(required_tweet_ids(ASSET_ROOT))
assert all(len(words) == len(set(words)) == 5 and all(word in attribute_vocab for word in words)
           for words in attributes.values())

class SarcasmDataset(Dataset):
    def __init__(self, split):
        rows = [ast.literal_eval(line) for line in
                (ASSET_ROOT / f'text/{split}.txt').read_text().splitlines() if line.strip()]
        self.rows = [row for row in rows if not blocked.intersection(row[1].split())]
        self.tokens = torch.zeros((len(self.rows), SEQ_LEN), dtype=torch.long)
        self.attributes = torch.empty((len(self.rows), 5), dtype=torch.long)
        self.labels = torch.tensor([row[-1] for row in self.rows], dtype=torch.float32)
        for i, row in enumerate(self.rows):
            token_ids = [word_vocab.get(token, word_vocab['<unk>']) for token in row[1].split()[:SEQ_LEN]]
            self.tokens[i, :len(token_ids)] = torch.tensor(token_ids, dtype=torch.long)
            self.attributes[i] = torch.tensor(
                [attribute_vocab.get(token, attribute_vocab['<unk>']) for token in attributes[str(row[0])]],
                dtype=torch.long)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        tweet_id = str(self.rows[index][0])
        image = torch.from_numpy(np.load(FEATURE_DIR / f'{tweet_id}.npy', allow_pickle=False))
        return self.tokens[index], image, self.attributes[index], self.labels[index], tweet_id

train_data, valid_data, test_data = (SarcasmDataset(split) for split in ('train', 'valid2', 'test2'))
generator = torch.Generator().manual_seed(SEEDS[0])
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', generator=generator,
                          persistent_workers=NUM_WORKERS > 0)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                         pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
positive = train_data.labels.sum().item()
positive_weight = (len(train_data) - positive) / positive
print('Train / validation / test:', len(train_data), len(valid_data), len(test_data))


Train / validation / test: 19816 2410 2409


## Hierarchical fusion model

In [6]:
class GuidedAttention(nn.Module):
    def __init__(self, raw_size, guidance_size):
        super().__init__()
        hidden_size = (raw_size + guidance_size) // 2
        self.raw = nn.Linear(raw_size, hidden_size)
        self.guidance = nn.Linear(guidance_size, hidden_size, bias=False)
        self.score = nn.Linear(hidden_size, 1)

    def forward(self, raw, guidance, mask=None):
        hidden = torch.tanh(self.raw(raw) + self.guidance(guidance).unsqueeze(1))
        scores = self.score(hidden)
        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(-1), float('-inf'))
        return scores.softmax(dim=1)

class HierarchicalFusion(nn.Module):
    def __init__(self, word_weights, attribute_weights):
        super().__init__()
        self.word_embedding = nn.Embedding.from_pretrained(word_weights.clone(), freeze=False, padding_idx=0)
        self.attribute_embedding = nn.Embedding.from_pretrained(attribute_weights.clone(), freeze=False)
        self.dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(200, 256, batch_first=True, bidirectional=True)
        self.initial_state = nn.Linear(200, 4 * 256)
        self.image_projection = nn.Linear(2048, 1024)
        self.attribute_attention = nn.Sequential(nn.Linear(200, 100), nn.ReLU(), nn.Linear(100, 1))
        sizes = [1024, 512, 200]
        self.reconstruction = nn.ModuleList([
            nn.ModuleList([GuidedAttention(m, n) for n in sizes]) for m in sizes])
        self.modality_scores = nn.ModuleList([
            nn.Sequential(nn.Linear(size, 512), nn.Tanh(), nn.Linear(512, 1)) for size in sizes])
        self.modality_projection = nn.ModuleList([
            nn.Sequential(nn.Linear(size, 512), nn.Tanh()) for size in sizes])
        self.classifier = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 1))
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.08, a=-0.16, b=0.16)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.01)
        nn.init.xavier_uniform_(self.image_projection.weight)
        for name, parameter in self.lstm.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(parameter)
            else:
                nn.init.zeros_(parameter)
                if 'bias_ih' in name:
                    with torch.no_grad():
                        parameter[256:512].fill_(1.0)

    def forward(self, tokens, images, attributes):
        lengths = tokens.ne(0).sum(dim=1)
        if (lengths == 0).any():
            raise ValueError('Encountered a text sequence containing only padding')
        text_mask = torch.arange(tokens.size(1), device=tokens.device)[None, :] < lengths[:, None]
        attribute_raw = self.attribute_embedding(attributes)
        attribute_weights = self.attribute_attention(attribute_raw).softmax(dim=1)
        attribute_guidance = (attribute_weights * attribute_raw).sum(dim=1)
        h_forward, h_backward, c_forward, c_backward = self.initial_state(attribute_guidance).relu().chunk(4, dim=1)
        embedded = self.dropout(self.word_embedding(tokens))
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed,
                                (torch.stack([h_forward, h_backward]),
                                 torch.stack([c_forward, c_backward])))
        text_raw, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=tokens.size(1))
        text_guidance = (text_raw * text_mask.unsqueeze(-1)).sum(dim=1) / lengths.unsqueeze(1)
        image_raw = self.image_projection(images).relu()
        raw = [image_raw, text_raw, attribute_raw]
        guidance = [image_raw.mean(dim=1), text_guidance, attribute_guidance]
        reconstructed = []
        for sequence, attentions, mask in zip(raw, self.reconstruction, [None, text_mask, None]):
            weights = torch.stack([attention(sequence, guide, mask)
                                   for attention, guide in zip(attentions, guidance)]).mean(dim=0)
            reconstructed.append((weights * sequence).sum(dim=1))
        scores = torch.cat([score(vector) for score, vector in
                            zip(self.modality_scores, reconstructed)], dim=1).softmax(dim=1)
        projected = torch.stack([projection(vector) for projection, vector in
                                 zip(self.modality_projection, reconstructed)], dim=1)
        fused = (scores.unsqueeze(-1) * projected).sum(dim=1)
        return self.classifier(fused).squeeze(-1)

    def regularization(self):
        return 0.5 * sum(parameter.square().sum() for name, parameter in self.named_parameters()
                         if parameter.requires_grad and name != 'word_embedding.weight')

@torch.no_grad()
def check_padding(model):
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device
    tokens = torch.tensor([[1, 2, 0], [3, 0, 0]], device=device)
    images = torch.zeros(2, 196, 2048, device=device)
    attributes = torch.zeros(2, 5, dtype=torch.long, device=device)
    padded = torch.cat([tokens, tokens.new_zeros(2, 4)], dim=1)
    torch.testing.assert_close(model(tokens, images, attributes), model(padded, images, attributes))
    mask = tokens.ne(0)
    raw = torch.zeros(2, 3, 512, device=device)
    for attention, size in zip(model.reconstruction[1], [1024, 512, 200]):
        weights = attention(raw, torch.zeros(2, size, device=device), mask).squeeze(-1)
        assert (weights[~mask] == 0).all(), 'Padding received attention weight'
        torch.testing.assert_close(weights.sum(dim=1), torch.ones(2, device=device))
    try:
        model(torch.zeros_like(tokens), images, attributes)
    except ValueError:
        pass
    else:
        raise AssertionError('All-padding text must be rejected')
    model.train(was_training)


## Train and evaluate

In [7]:
def binary_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    predicted = np.asarray(probabilities) > 0.5
    tp = int(((labels == 1) & predicted).sum()); fp = int(((labels == 0) & predicted).sum())
    tn = int(((labels == 0) & ~predicted).sum()); fn = int(((labels == 1) & ~predicted).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1,
            'accuracy': (tp + tn) / len(labels)}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    labels, probabilities, tweet_ids = [], [], []
    for tokens, images, attributes, targets, ids in tqdm(loader, desc='Evaluate', leave=False):
        logits = model(tokens.to(DEVICE, non_blocking=True), images.to(DEVICE, non_blocking=True),
                       attributes.to(DEVICE, non_blocking=True))
        probabilities.extend(logits.sigmoid().cpu().tolist())
        labels.extend(targets.tolist()); tweet_ids.extend(ids)
    return binary_metrics(labels, probabilities), tweet_ids, labels, probabilities

RUN_ROOT.mkdir(parents=True, exist_ok=True)
all_results = []
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    train_loader = DataLoader(
        train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda',
        generator=torch.Generator().manual_seed(seed), persistent_workers=NUM_WORKERS > 0)
    run_dir = RUN_ROOT / f'seed_{seed}'
    run_dir.mkdir(exist_ok=False)
    feature_manifests = [json.loads(path.read_text()) for path in sorted((FEATURE_DIR / 'manifests').glob('*.json'))]
    config = {'model': 'hierarchical_fusion', 'variant': 'finetuned_features_clip_sentiment_packed',
              'attribute_manifest': attribute_manifest,
              'seed': seed, 'attribute_predictions': str(ATTRIBUTE_PREDICTIONS), 'attribute_embedding': 'trainable',
              'text_padding': 'packed_lstm_masked_mean_masked_attention',
              'batch_size': BATCH_SIZE, 'max_epochs': EPOCHS, 'patience': PATIENCE,
              'learning_rate': LEARNING_RATE, 'lr_decay_per_epoch': 0.8, 'l2': L2,
              'sequence_length': SEQ_LEN, 'hidden_size': 256, 'input_dropout': 0.2, 'gradient_clipping': 5.0,
              'positive_weight': positive_weight, 'checkpoint_selection': 'validation positive-class F1',
              'asset_root': str(ASSET_ROOT), 'feature_dir': str(FEATURE_DIR),
              'split_counts': {'train': len(train_data), 'valid': len(valid_data), 'test': len(test_data)},
              'upstream_revision': 'f42b16510208624d91fa545ca9bb64c6335f971e',
              'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__,
              'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
              'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
              'device': str(DEVICE), 'num_workers': NUM_WORKERS, 'feature_manifests': feature_manifests}
    (run_dir / 'config.json').write_text(json.dumps(config, indent=2) + '\n')
    base_model = HierarchicalFusion(word_weights, attribute_weights).to(DEVICE)
    check_padding(base_model)
    network = nn.DataParallel(base_model) if torch.cuda.device_count() > 1 else base_model
    print(f'Seed {seed} | GPUs: {torch.cuda.device_count()}')
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=DEVICE))
    optimizer = torch.optim.Adam(base_model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.8)
    best_f1, stale_epochs, history = -1.0, 0, []
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        network.train(); epoch_loss, seen = 0.0, 0
        epoch_start = time.perf_counter()
        for tokens, images, attributes, labels, _ in tqdm(train_loader, desc=f'Seed {seed} epoch {epoch}/{EPOCHS}'):
            tokens, images = tokens.to(DEVICE, non_blocking=True), images.to(DEVICE, non_blocking=True)
            attributes, labels = attributes.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(network(tokens, images, attributes), labels) + L2 * base_model.regularization()
            if not torch.isfinite(loss): raise FloatingPointError('Non-finite training loss')
            loss.backward(); nn.utils.clip_grad_norm_(base_model.parameters(), 5.0); optimizer.step()
            epoch_loss += loss.item() * len(labels); seen += len(labels)
        valid_metrics, _, _, _ = evaluate(network, valid_loader)
        record = {'epoch': epoch, 'train_loss': epoch_loss / seen, 'validation': valid_metrics,
                  'seconds': time.perf_counter() - epoch_start}
        history.append(record)
        (run_dir / 'history.json').write_text(json.dumps(history, indent=2) + '\n')
        print(json.dumps({'seed': seed, **record}))
        if valid_metrics['f1'] > best_f1:
            best_f1, stale_epochs = valid_metrics['f1'], 0
            torch.save({'model': base_model.state_dict(), 'epoch': epoch,
                        'validation_f1': best_f1, 'seed': seed}, run_dir / 'best.pt.tmp')
            os.replace(run_dir / 'best.pt.tmp', run_dir / 'best.pt')
        else:
            stale_epochs += 1
        scheduler.step()
        if stale_epochs >= PATIENCE: break
    training_seconds = time.perf_counter() - start
    checkpoint = torch.load(run_dir / 'best.pt', map_location=DEVICE, weights_only=True)
    base_model.load_state_dict(checkpoint['model'])
    test_start = time.perf_counter()
    test_metrics, ids, labels, probabilities = evaluate(network, test_loader)
    result = {'seed': seed, 'selected_epoch': checkpoint['epoch'],
              'best_validation_f1': checkpoint['validation_f1'], 'test': test_metrics,
              'training_seconds': training_seconds, 'test_seconds': time.perf_counter() - test_start,
              'trainable_parameters': sum(p.numel() for p in base_model.parameters() if p.requires_grad)}
    (run_dir / 'results.json').write_text(json.dumps(result, indent=2) + '\n')
    with (run_dir / 'test_predictions.csv').open('w', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['tweet_id', 'label', 'probability', 'prediction'])
        writer.writerows((tweet_id, int(label), probability, int(probability > 0.5))
                         for tweet_id, label, probability in zip(ids, labels, probabilities))
    all_results.append(result)
    print(json.dumps(result, indent=2))
    del network, base_model, optimizer, scheduler, criterion
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

summary = {metric: {'mean': float(np.mean([r['test'][metric] for r in all_results])),
                    'std': float(np.std([r['test'][metric] for r in all_results], ddof=1))}
           for metric in ('precision', 'recall', 'f1', 'accuracy')}
summary['seeds'] = all_results
(RUN_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))


Seed 42 | GPUs: 2


Seed 42 epoch 1/8:   0%|          | 0/619 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"seed": 42, "epoch": 1, "train_loss": 0.43056431802678763, "validation": {"precision": 0.7896341463414634, "recall": 0.8102189781021898, "f1": 0.7997941327843541, "accuracy": 0.8385892116182573}, "seconds": 394.53725080000004}


Seed 42 epoch 2/8:   0%|          | 0/619 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

... (63 outputs left out)

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"seed": 44, "epoch": 8, "train_loss": 0.017016215834793708, "validation": {"precision": 0.806282722513089, "recall": 0.8029197080291971, "f1": 0.8045977011494253, "accuracy": 0.8448132780082988}, "seconds": 151.72985994099963}


Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{
  "seed": 44,
  "selected_epoch": 3,
  "best_validation_f1": 0.8140133951571354,
  "test": {
    "precision": 0.7843719090009891,
    "recall": 0.8269030239833159,
    "f1": 0.8050761421319796,
    "accuracy": 0.8405977584059776
  },
  "training_seconds": 1136.8352792880005,
  "test_seconds": 27.73581883999941,
  "trainable_parameters": 14909926
}
{
  "precision": {
    "mean": 0.7620584228132609,
    "std": 0.019371754290502138
  },
  "recall": {
    "mean": 0.8498435870698645,
    "std": 0.02082897221630678
  },
  "f1": {
    "mean": 0.8032394197338496,
    "std": 0.003835468244004774
  },
  "accuracy": {
    "mean": 0.8342327383423275,
    "std": 0.006048842671757123
  },
  "seeds": [
    {
      "seed": 42,
      "selected_epoch": 2,
      "best_validation_f1": 0.8175542406311637,
      "test": {
        "precision": 0.7495429616087751,
        "recall": 0.8550573514077163,
        "f1": 0.7988309790550414,
        "accuracy": 0.8285595682855957
      },
      "training_seconds":